# Day 8 — Chatbot
### AI that remembers your conversation!
---
**Model:** Qwen/Qwen3-0.6B

**The big difference from Day 7:**
- Day 7 = AI generates text (no memory)
- Day 8 = AI has a conversation (remembers everything you said)

> 💡 **Tip:** This small model runs on any laptop (CPU is fine). A GPU just makes replies faster.


---
## Step 1 — Install and load the chatbot


In [1]:
!pip install -U transformers gradio accelerate -q
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers.utils import logging
import warnings
warnings.filterwarnings('ignore')
logging.set_verbosity_error()

MODEL = 'Qwen/Qwen3-0.6B'

# Use GPU if available, otherwise CPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'
dtype  = torch.float16 if device == 'cuda' else torch.float32
print('Running on:', device)

print('Loading chatbot model...')
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model     = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=dtype).to(device)
print('Chatbot ready!')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 45.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.5/64.5 kB 3.7 MB/s eta 0:00:00
Running on: cpu
Loading chatbot model...


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.50GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Chatbot ready!


---
## Step 2 — Create the chat function

> We store the conversation as a simple list.
> Every message (yours + bot) gets added to that list.
> The bot reads the whole list each time — that is its memory!


In [2]:
def chat(user_message, history):
    '''
    user_message : what you typed
    history      : list of all messages so far
    '''
    # Add your message to history
    history.append({'role': 'user', 'content': user_message})

    # Build the full conversation for the model
    # Qwen has a built-in chat format — the tokenizer applies it for us
    # enable_thinking=False -> direct answers (no long 'thinking' text)
    messages = [
        {'role': 'system', 'content': 'You are a helpful and friendly AI assistant.'}
    ] + history

    text = tokenizer.apply_chat_template(
        messages,
        tokenize              = False,
        add_generation_prompt = True,
        enable_thinking       = False
    )

    # Tokenize and generate reply
    inputs = tokenizer(text, return_tensors='pt').to(model.device)
    output = model.generate(
        **inputs,
        max_new_tokens = 150,
        do_sample      = True,
        temperature    = 0.7,
        top_p          = 0.8,
        top_k          = 20
    )

    # Keep only the NEW tokens (the bot's reply)
    new_tokens = output[0][inputs['input_ids'].shape[1]:]
    bot_reply  = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    # Add bot reply to history
    history.append({'role': 'assistant', 'content': bot_reply})

    return bot_reply, history

print('chat() function ready!')


chat() function ready!


---
## Step 3 — Send one message


In [3]:
# Start with empty history
history = []

# Send first message
reply, history = chat('What is artificial intelligence?', history)

print('You :', 'What is artificial intelligence?')
print('Bot :', reply)


You : What is artificial intelligence?
Bot : Artificial Intelligence (AI) refers to the simulation of human intelligence tasks that are carried out by machines. These tasks include problem-solving, learning, reasoning, understanding language, and making decisions. AI systems are designed to perform tasks that would typically require human intelligence, such as analyzing data, identifying patterns, and making predictions. AI is a broad field that encompasses a variety of techniques and models, including machine learning, natural language processing, and computer vision.


---
## Step 4 — See the memory in action!

> Tell the bot something personal.
> Then ask a follow-up — it should remember!


In [4]:
# Fresh start
history = []

# Message 1 — tell it something
reply1, history = chat('My favourite sport is cricket.', history)
print('You : My favourite sport is cricket.')
print('Bot :', reply1)
print()

# Message 2 — ask a follow-up
reply2, history = chat('What sport did I just mention?', history)
print('You : What sport did I just mention?')
print('Bot :', reply2)
print()
print('The bot should remember cricket!')


You : My favourite sport is cricket.
Bot : Great! Cricket is a fantastic sport! I love watching it too. Let me know if you want to learn more about it or have any questions! 🏰✨

You : What sport did I just mention?
Bot : I'm not sure what sport you just mentioned. Could you clarify or rephrase? 😊

The bot should remember cricket!


---
## Step 5 — Have a full conversation


In [5]:
# Fresh start
history = []

questions = [
    'Tell me about smart cities.',
    'What sensors are used in smart cities?',
    'How does AI help manage traffic?',
    'What are the benefits for citizens?',
]

print('Full conversation about Smart Cities:')
print('=' * 45)
print()

for i, question in enumerate(questions, 1):
    reply, history = chat(question, history)
    print(f'Turn {i}')
    print(f'You : {question}')
    print(f'Bot : {reply}')
    print()


Full conversation about Smart Cities:

Turn 1
You : Tell me about smart cities.
Bot : Smart cities are urban areas that leverage technology, data, and digital infrastructure to improve the quality of life for residents, enhance public services, and promote sustainable development. Here are some key aspects of smart cities:

### 1. **Smart Infrastructure**
   - **IoT (Internet of Things)**: Sensors and devices collect real-time data from various sources (e.g., traffic, energy, waste management).
   - **Smart Grids**: Efficient energy distribution using renewable sources and automated systems.
   - **Smart Transportation**: Traffic management systems, public transit optimization, and bike-sharing networks.

### 2. **Data-Driven Decision Making**
   - **Big Data**: Analyzing data from cities to predict issues (e.g., traffic congestion, energy

Turn 2
You : What sensors are used in smart cities?
Bot : Smart cities utilize a variety of sensors to monitor and manage urban environments effect

---
## Step 6 — Try your own topic!

> ✏️ Change the questions to anything you like!


In [6]:
# ✏️ Change these questions!
history = []

my_questions = [
    'Tell me about machine learning.',
    'How is it different from regular programming?',
    'Can you give me a simple example?',
]

for i, question in enumerate(my_questions, 1):
    reply, history = chat(question, history)
    print(f'You : {question}')
    print(f'Bot : {reply}')
    print()


You : Tell me about machine learning.
Bot : Machine learning is a subset of artificial intelligence that focuses on building systems that can learn patterns from data without being explicitly programmed. These systems can analyze and learn from historical data, identify trends, and make predictions or decisions based on that learning.

### Key Concepts:
1. **Supervised Learning**: The model learns to map inputs to outputs (e.g., predicting house prices based on features).
2. **Unsupervised Learning**: The model learns to find hidden patterns or structures in data (e.g., clustering customers into groups).
3. **Reinforcement Learning**: The model learns through trial and error, receiving rewards for achieving specific goals.

### Applications:
- **Healthcare**: Diagnosing diseases, predicting patient outcomes.
- **Finance**: Fraud

You : How is it different from regular programming?
Bot : Machine learning is fundamentally different from regular programming in several key ways:

1. **No E

---
## Step 7 — Build a Gradio Chatbot App

> Type any message — bot replies and remembers!
> Click **Clear** to start a fresh conversation.


In [9]:
import gradio as gr
import inspect

# Store history between messages
memory = {'history': []}

def chat_app(user_message, chatbox):
    if not user_message.strip():
        return chatbox, ''

    # Get reply using our chat() function
    reply, memory['history'] = chat(user_message, memory['history'])

    # Add to display
    chatbox.append({'role': 'user',      'content': user_message})
    chatbox.append({'role': 'assistant', 'content': reply})
    return chatbox, ''


def clear_chat():
    # Reset memory — fresh start
    memory['history'] = []
    return [], ''


# Works with both older and newer Gradio versions
chatbot_kwargs = {}
if 'type' in inspect.signature(gr.Chatbot.__init__).parameters:
    chatbot_kwargs['type'] = 'messages'

with gr.Blocks(title='AI Chatbot') as app:
    gr.Markdown('# AI Chatbot — Qwen3')
    gr.Markdown('Chat with AI! It **remembers** your whole conversation.')
    gr.Markdown('Click **Clear** to start fresh.')

    chatbox   = gr.Chatbot(label='Conversation', height=350, **chatbot_kwargs)
    msg_input = gr.Textbox(
        label       = 'Your message',
        placeholder = 'Type anything...',
        lines       = 1
    )
    with gr.Row():
        send_btn  = gr.Button('Send',  variant='primary')
        clear_btn = gr.Button('Clear')

    send_btn.click(
        fn      = chat_app,
        inputs  = [msg_input, chatbox],
        outputs = [chatbox, msg_input]
    )
    clear_btn.click(
        fn      = clear_chat,
        outputs = [chatbox, msg_input]
    )

app.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://fa5824bdf424da14c1.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [8]:
app.close()
print('App closed.')


Closing server running on port: 7860
App closed.


---
## What we learned

| Step | What we did |
|------|-------------|
| 1 | Loaded Qwen3-0.6B model directly |
| 2 | Created chat() function with history list |
| 3 | Sent one message and got a reply |
| 4 | Saw memory in action |
| 5 | Had a full 4-turn conversation |
| 6 | Tried our own topic |
| 7 | Built Gradio chatbot app |

### Key words:
- **`history`** — a list that stores all messages
- **Memory** — passing the same history list to every message
- **Clear** — resetting history to empty list `[]`

**Tomorrow — Day 9:** Translation + Summarization!
